In [84]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from astropy.io import fits
import glob
from sft import *
from datetime import datetime, timedelta
from interpolation import interp1d
from utils import *
import sunpy.visualization.colormaps as cm

In [102]:
def show(data):
    x_lims = mdates.date2num([datetime(2024,1,1), datetime(2026,1,1)])
    y_lims = [-1, 1]

    fig, ax = plt.subplots(figsize=(14,9))
    ax.imshow(data, cmap='seismic', vmin=-10, vmax=10, #interpolation='bicubic',
              extent=[x_lims[0], x_lims[1], y_lims[0], y_lims[1]], aspect='auto', origin='lower')

    ax.set_yticks(np.sin(np.arange(-90, 91, 15) * np.pi / 180), np.arange(-90, 91, 15))

    ax.xaxis_date()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y/%m'))

    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

In [2]:
files_hmi = np.array(sorted(glob.glob('/home/ulyanov/data/sdo/hmi/synoptic_maps/*')))

data_hmi = []
dates_hmi = []

for file in files_hmi:
    with fits.open(file) as hdul:
        header = hdul[0].header.copy()
        data = hdul[0].data.copy()

    data = rebin(data, 2)
    data_hmi.append(data)

    date = datetime.fromisoformat(header['T_OBS'][:-4].replace('.', '').replace('_', 'T'))
    dates_hmi.append(date)

data_hmi = np.array(data_hmi)
dates_hmi = np.array(dates_hmi)

flux_hmi = np.nanmean(data_hmi, axis=-1)
flux_hmi = (flux_hmi[:,1:] + flux_hmi[:,:-1]) / 2
flux_hmi = np.nan_to_num(flux_hmi)

/tmp/ipykernel_4806/4212245791.py:20: RuntimeWarning: Mean of empty slice
  flux_hmi = np.nanmean(data_hmi, axis=-1)


In [3]:
files = np.array(sorted(glob.glob('/home/ulyanov/data/solo/phi/synoptic_maps/*')))

data_fdt = []
weights = []
dates_fdt = []

for file in files:
    s = np.load(file)
    mean = s['mean']
    weight = s['weight']
    lati = s['lat']

    start = datetime.fromisoformat(str(s['start']))
    end = datetime.fromisoformat(str(s['end']))
    delta = end - start
    date = start + delta / 2

    data_fdt.append(mean)
    weights.append(weight)
    dates_fdt.append(date)

data_fdt = np.array(data_fdt)
weights = np.array(weights)
dates_fdt = np.array(dates_fdt)

weights = weights.clip(0,0.1) / 0.1
data_fdt = np.nan_to_num(data_fdt * weights) + np.nan_to_num(data_hmi * (1 - np.nan_to_num(weights)))

flux_fdt = np.nanmean(data_fdt, axis=-1)
flux_fdt = np.nan_to_num(flux_fdt)
flux_fdt = (flux_fdt[:,1:] + flux_fdt[:,:-1]) / 2

In [4]:
dsine = 1 / 359.5
lati = np.arange(-1,1 + dsine / 2, dsine)
lati = np.arcsin(lati.clip(-1,1)) * 180 / np.pi
lat = (lati[1:] + lati[:-1]) / 2

In [94]:
plt.figure(figsize=(10,8))
plt.imshow(data_fdt[0], aspect='auto', origin='lower', cmap='hmimag', vmin=-1000, vmax=1000)
plt.tight_layout()

In [6]:
plt.figure(figsize=(10,8))
plt.plot(lat, flux_fdt[23])
plt.plot(lat, flux_hmi[23])
plt.xlim(-90,90)
plt.ylim(-10,10)
plt.grid(True)
plt.tight_layout()

In [103]:
show(flux_fdt.T)

In [104]:
show(flux_hmi.T)

In [41]:
R = 695e8
u = 1200
d = 500e10
lam = 45
dt = timedelta(days=1).total_seconds()

xi = lati * np.pi / 180 * R
vi = u * flow(lati, a=0.5)
li = 2 * np.pi * R * np.cos(lati  * np.pi / 180)

y = flux_fdt[0].copy()

Q = []
T = []

for t in np.arange(dates_fdt[0], dates_fdt[-1], timedelta(days=1)):
    y_ = interp1d(flux_fdt, dates_fdt, t)
    y = advect(y, xi, vi, dt, li)
    y = diffuse(y, xi, d, dt, li)
    y[np.abs(lat) < lam] = y_[np.abs(lat) < lam]

    Q.append(y)
    T.append(t)

Q = np.array(Q)
T = np.array(T)

In [105]:
show(Q.T)

In [46]:
i = 23

plt.figure(figsize=(10,8))
plt.plot(lat, flux_fdt[0])
plt.plot(lat, flux_fdt[i])
plt.plot(lat, interp1d(Q, T, dates_fdt[i]))
plt.xlim(-90,90)
plt.ylim(-15,15)
plt.grid(True)
plt.tight_layout()

In [49]:
?plt.xticks
